# Experiment: WOP Python vs C++ Benchmark on Unit Cube

Objective:
- Compare numerical outputs and runtime of Python and C++ implementations of WOP.
- Run on unit cube exterior for `N = [100, 1000, 10000, 100000, 1000000]`.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

try:
    import pandas as pd
    HAVE_PANDAS = True
except ModuleNotFoundError:
    pd = None
    HAVE_PANDAS = False


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "wop").exists():
            return candidate
    raise RuntimeError("Cannot find project root with pyproject.toml and wop package.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from wop.geometry import make_axis_aligned_box, orient_normals
from wop.wop import estimate_wop

N_VALUES = [100, 1000, 10000, 100000, 1000000]
X0 = np.array([3.0, 0.0, 0.0], dtype=float)
BOX_MIN = np.array([-1.0, -1.0, -1.0], dtype=float)
BOX_MAX = np.array([1.0, 1.0, 1.0], dtype=float)
INTERIOR = np.array([0.0, 0.0, 0.0], dtype=float)
A = np.array([0.2, -0.1, 0.3], dtype=float)
MAX_STEPS = 200_000
R_MAX = 1e6
BASE_SEED = 314159

POLY = orient_normals(make_axis_aligned_box(BOX_MIN, BOX_MAX), INTERIOR)


def exact_u(x: np.ndarray) -> float:
    return float(1.0 / np.linalg.norm(x - A))


def boundary_f(y: np.ndarray, _face: int | None) -> float:
    return exact_u(y)


def find_cpp_cli(project_root: Path) -> Path:
    env_path = os.environ.get("WOP_CPP_CLI")
    if env_path:
        p = Path(env_path).expanduser().resolve()
        if p.exists():
            return p
        raise FileNotFoundError(f"WOP_CPP_CLI is set but does not exist: {p}")

    candidates = [
        project_root / "cpp" / "build" / "Release" / "wop_cli.exe",
        project_root / "cpp" / "build" / "Debug" / "wop_cli.exe",
        project_root / "cpp" / "build" / "wop_cli.exe",
        project_root / "cpp" / "build" / "wop_cli",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Cannot find C++ CLI binary. Build it first or set WOP_CPP_CLI env var."
    )


CPP_CLI = find_cpp_cli(PROJECT_ROOT)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"CPP_CLI = {CPP_CLI}")
print(f"HAVE_PANDAS = {HAVE_PANDAS}")
if not HAVE_PANDAS:
    print("pandas is not installed; notebook will use fallback table objects.")


## Benchmark helpers

Both paths run the same unit-cube setup and return comparable metrics including runtime.


In [ ]:
def run_python_once(n_paths: int, seed: int) -> dict[str, float | int | str]:
    rng = np.random.default_rng(seed)
    t0 = time.perf_counter()
    res = estimate_wop(
        poly=POLY,
        x0=X0,
        boundary_f=boundary_f,
        n_paths=n_paths,
        rng=rng,
        max_steps=MAX_STEPS,
        u_inf=0.0,
        r_max=R_MAX,
    )
    elapsed = time.perf_counter() - t0
    exact = exact_u(X0)
    return {
        "engine": "python",
        "N": int(n_paths),
        "seed": int(seed),
        "J": float(res.J),
        "S2": float(res.S2),
        "eps": float(res.eps),
        "exact": float(exact),
        "abs_error": float(abs(res.J - exact)),
        "n_total": int(res.n_total),
        "n_truncated": int(res.n_truncated),
        "mean_steps": float(res.mean_steps),
        "runtime_sec": float(elapsed),
    }


def run_cpp_once(n_paths: int, seed: int) -> dict[str, float | int | str]:
    cmd = [
        str(CPP_CLI),
        "--example",
        "box",
        "--x0",
        f"{X0[0]} {X0[1]} {X0[2]}",
        "--n",
        str(int(n_paths)),
        "--seed",
        str(int(seed)),
        "--max-steps",
        str(int(MAX_STEPS)),
        "--r-max",
        str(float(R_MAX)),
        "--json",
    ]
    t0 = time.perf_counter()
    out = subprocess.check_output(cmd, text=True, cwd=PROJECT_ROOT)
    elapsed = time.perf_counter() - t0
    data = json.loads(out)
    return {
        "engine": "cpp",
        "N": int(n_paths),
        "seed": int(seed),
        "J": float(data["J"]),
        "S2": float(data["S2"]),
        "eps": float(data["eps"]),
        "exact": float(data["exact"]),
        "abs_error": float(data["abs_error"]),
        "n_total": int(data["n_total"]),
        "n_truncated": int(data["n_truncated"]),
        "mean_steps": float(data["mean_steps"]),
        "runtime_sec": float(elapsed),
    }


## Run benchmark on N grid


In [ ]:
rows: list[dict[str, float | int | str]] = []

for idx, n in enumerate(N_VALUES):
    seed = BASE_SEED + idx
    print(f"Running N={n:,}, seed={seed} ...")
    rows.append(run_python_once(n, seed))
    rows.append(run_cpp_once(n, seed))

if HAVE_PANDAS:
    results_df = pd.DataFrame(rows).sort_values(["N", "engine"]).reset_index(drop=True)
    results_df["runtime_ms"] = 1e3 * results_df["runtime_sec"]
    display(results_df)
else:
    results_df = rows
    results_df


## Comparative table (values + runtime)


In [ ]:
def build_compare_rows(raw_rows: list[dict[str, float | int | str]]) -> list[dict[str, float | int]]:
    by_engine_and_n: dict[tuple[str, int], dict[str, float | int | str]] = {}
    for row in raw_rows:
        by_engine_and_n[(str(row["engine"]), int(row["N"]))] = row

    compare_rows: list[dict[str, float | int]] = []
    for n in N_VALUES:
        py = by_engine_and_n[("python", n)]
        cpp = by_engine_and_n[("cpp", n)]
        compare_rows.append(
            {
                "N": n,
                "J_py": float(py["J"]),
                "J_cpp": float(cpp["J"]),
                "delta_J_abs": abs(float(py["J"]) - float(cpp["J"])),
                "abs_error_py": float(py["abs_error"]),
                "abs_error_cpp": float(cpp["abs_error"]),
                "eps_py": float(py["eps"]),
                "eps_cpp": float(cpp["eps"]),
                "runtime_sec_py": float(py["runtime_sec"]),
                "runtime_sec_cpp": float(cpp["runtime_sec"]),
                "speedup_cpp_vs_python": float(py["runtime_sec"]) / float(cpp["runtime_sec"]),
                "mean_steps_py": float(py["mean_steps"]),
                "mean_steps_cpp": float(cpp["mean_steps"]),
                "n_truncated_py": int(py["n_truncated"]),
                "n_truncated_cpp": int(cpp["n_truncated"]),
            }
        )
    return compare_rows


if HAVE_PANDAS:
    py_df = (
        results_df[results_df["engine"] == "python"]
        .set_index("N")
        .add_suffix("_py")
    )
    cpp_df = (
        results_df[results_df["engine"] == "cpp"]
        .set_index("N")
        .add_suffix("_cpp")
    )

    compare_df = py_df.join(cpp_df, how="inner")
    compare_df["delta_J_abs"] = (compare_df["J_py"] - compare_df["J_cpp"]).abs()
    compare_df["delta_eps_abs"] = (compare_df["eps_py"] - compare_df["eps_cpp"]).abs()
    compare_df["speedup_cpp_vs_python"] = compare_df["runtime_sec_py"] / compare_df["runtime_sec_cpp"]

    final_columns = [
        "J_py",
        "J_cpp",
        "delta_J_abs",
        "abs_error_py",
        "abs_error_cpp",
        "eps_py",
        "eps_cpp",
        "runtime_sec_py",
        "runtime_sec_cpp",
        "speedup_cpp_vs_python",
        "mean_steps_py",
        "mean_steps_cpp",
        "n_truncated_py",
        "n_truncated_cpp",
    ]

    compare_df = compare_df[final_columns]
    display(compare_df.round(6))
else:
    compare_rows = build_compare_rows(results_df)
    compare_rows


## Notes

- `runtime_sec_*` is measured wall-clock time per single run.
- For more stable timing, repeat each run several times and aggregate median time.
- C++ binary path can be overridden with `WOP_CPP_CLI`.
- If `pandas` is installed, outputs are returned as DataFrames; otherwise fallback list-of-dicts is used.
